# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and the fields (columns) within them. All entities are referenced by their `@id`s according to the Croissant schema.

In [ ]:
# List all available record sets and fields by @id
record_sets = list(dataset.metadata.record_sets)
print(f"Record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet Name: {rs.name}\n  @id: {rs.id}\n  Description: {getattr(rs, 'description', None)}")
    if hasattr(rs, 'fields'):
        print("  Fields (by @id):")
        for field in rs.fields:
            print(f"    - {field.id} ({field.name}, type={field.data_type})")
    print('-' * 60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below we extract all record sets into dataframes, keyed by their `@id`.

In [ ]:
# Extract data for each record set by @id.
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading data for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Number of rows: {len(df)}; Columns: {df.columns.tolist()}")
    print('-' * 60)

# Choose one record set for focused analysis (typically the main table)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Main record set: {main_record_set_id}\nColumns: {dataframes[main_record_set_id].columns.tolist()}")
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We'll work on one numeric field and show filtering and normalization using field `@id`s.

In [ ]:
# Select a numeric field for analysis by @id. Replace with an actual numeric field @id from section 2.
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]

    # Heuristically choose the first numeric-like column
    numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    print("Numeric fields detected:", numeric_fields)
    numeric_field_id = numeric_fields[0] if len(numeric_fields) > 0 else None

    if numeric_field_id:
        threshold = df[numeric_field_id].mean()  # Use the mean for demonstration
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to group by a categorical field
        categorical_cols = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if len(categorical_cols) > 0:
            group_field = categorical_cols[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
            print(f"Grouped mean {numeric_field_id} by {group_field}:")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we'll plot the distribution of the selected numeric field and mean values by a grouped categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouped_df from EDA exists
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10,4))
        grouped_df.sort_values(numeric_field_id).plot(kind='bar', legend=False, ax=plt.gca())
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field)
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 mass spectrometry dataset by programmatically loading Croissant metadata and records using their `@id`. We previewed the available record sets and fields, loaded the main dataset, filtered and normalized a numeric field, and visualized its distribution. For deeper exploration, use the provided `@id`s to flexibly reference all data elements across the Croissant schema.